In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from math import pi

# Rutas del proyecto
DATA_DIR = Path("data")
FIG_DIR = DATA_DIR / "graficos"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def buscar_csv(nombre):
    """Busca primero en data/ y después en el directorio actual."""
    for ruta in (DATA_DIR / nombre, Path(nombre)):
        if ruta.exists():
            return ruta
    return DATA_DIR / nombre

# 1. Configuración estética académica general
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# 2. Cargar datos
df = pd.read_csv(buscar_csv("resultados_finales_v2.csv"))

# 3. Forzar tipo numérico en las columnas necesarias por si acaso
score_cols = [
    'juez_p1_andamiaje', 'juez_p1_coherencia',
    'juez_p2_andamiaje', 'juez_p2_coherencia',
    'juez_medio_integracion', 'juez_bajo_integracion',
    'juez_fidelidad'
]
sim_cols = [
    'sim_pista_1', 'sim_pista_2',
    'sim_enunciado_medio', 'sim_enunciado_bajo'
]

for col in score_cols + sim_cols + ['tiempo_respuesta']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['sim_global'] = df[sim_cols].mean(axis=1)
df['nota_global_juez'] = df[score_cols].mean(axis=1)

print(f"Datos cargados correctamente. {len(df)} filas y {df['modelo_usado'].nunique()} modelos.")


In [ ]:
# Calcular las medias agrupando por modelo
tabla = df.groupby('modelo_usado').agg(
    Tareas_Completadas=('id', 'count'),
    Tiempo_Medio_s=('tiempo_respuesta', 'mean'),
    Similitud_Media=('sim_enunciado_bajo', 'mean'),
    P1_Andamiaje_Media=('juez_p1_andamiaje', 'mean'),
    Integ_Bajo_Media=('juez_bajo_integracion', 'mean'),
    Fidelidad_Media=('juez_fidelidad', 'mean'),
    Nota_Global_Media=('nota_global_juez', 'mean')
).reset_index()

# Redondear a 2 decimales para que quede elegante en la memoria
tabla = tabla.round(2)

# Ordenar del mejor al peor según la nota global
tabla = tabla.sort_values(by='Nota_Global_Media', ascending=False)

# Mostrar la tabla en el notebook
display(tabla)



In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Ordenar los modelos de mayor a menor nota
orden_modelos = df.groupby('modelo_usado')['nota_global_juez'].mean().sort_values(ascending=False).index

sns.barplot(
    data=df, 
    x='nota_global_juez', 
    y='modelo_usado', 
    order=orden_modelos, 
    palette='viridis', 
    errorbar='ci', # Muestra el intervalo de confianza (estabilidad)
    capsize=0.1,
    ax=ax
)

ax.set_xlabel("Nota Media Global del Juez (1-5)")
ax.set_ylabel("Modelo Evaluado")

plt.tight_layout()
plt.savefig(FIG_DIR / "ranking_notas_juez.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# HEATMAP DE PUNTUACIONES DEL JUEZ POR DIMENSIÓN
# ==========================================

nombres_dimensiones = {
    'juez_p1_andamiaje': 'P1 Andamiaje',
    'juez_p1_coherencia': 'P1 Coherencia',
    'juez_p2_andamiaje': 'P2 Andamiaje',
    'juez_p2_coherencia': 'P2 Coherencia',
    'juez_medio_integracion': 'Integ. Medio',
    'juez_bajo_integracion': 'Integ. Bajo',
    'juez_fidelidad': 'Fidelidad'
}

score_data = df.groupby('modelo_usado')[score_cols].mean()
score_data = score_data.loc[orden_modelos]
score_data = score_data.rename(columns=nombres_dimensiones)

fig, ax = plt.subplots(figsize=(11, 6))

sns.heatmap(
    score_data,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=1,
    vmax=5,
    cbar_kws={'label': 'Valor medio (1-5)'},
    linewidths=0.5,
    linecolor='white',
    ax=ax
)

ax.set_xlabel("Dimensión Pedagógica Evaluada")
ax.set_ylabel("Modelo Evaluado")

plt.tight_layout()
plt.savefig(FIG_DIR / "heatmap_juez_dimensiones.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Agrupar la media de similitud por cada columna para cada modelo
sim_data = df.groupby('modelo_usado')[sim_cols].mean()

# Limpiar los nombres de las columnas para que queden bonitos en el gráfico
sim_data.columns = ['Pista 1', 'Pista 2', 'Enunciado Medio', 'Enunciado Bajo']

# Ordenar por la media global de similitud de mayor a menor
sim_data['Media'] = sim_data.mean(axis=1)
sim_data = sim_data.sort_values('Media', ascending=False).drop(columns=['Media'])

# Crear el Heatmap
sns.heatmap(
    sim_data, 
    annot=True,        # Mostrar el número dentro de la celda
    fmt=".3f",         # 3 decimales
    cmap="YlGnBu",     # Paleta de amarillo (bajo) a azul oscuro (alto)
    vmin=0.0, vmax=1.0,# Forzar la escala de 0 a 1
    cbar_kws={'label': 'Similitud del Coseno'},
    ax=ax
)

ax.set_ylabel("Modelo Evaluado")
ax.set_xlabel("Componente Evaluado")

plt.tight_layout()
plt.savefig(FIG_DIR / "heatmap_similitud.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Usamos todos los datos (cada punto es una tarea de un modelo)
sns.scatterplot(
    data=df, 
    x='sim_global', 
    y='nota_global_juez', 
    hue='modelo_usado', 
    alpha=0.7, # Ligeramente transparentes para ver si se solapan
    s=100,
    palette='Set2',
    ax=ax
)

# Añadir una línea de tendencia general para guiar el ojo
sns.regplot(
    data=df, 
    x='sim_global', 
    y='nota_global_juez', 
    scatter=False, 
    color='gray', 
    line_kws={'linestyle':'--'},
    ax=ax
)

ax.set_xlabel("Similitud Semántica Global (0 a 1)")
ax.set_ylabel("Nota Global del Juez (1-5)")

ax.legend(title="Modelo Evaluado", bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)

plt.tight_layout()
plt.savefig(FIG_DIR / "scatter_similitud_vs_juez.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Agrupar las medias por modelo
agrupado = df.groupby('modelo_usado')[['tiempo_respuesta', 'nota_global_juez']].mean().reset_index()

sns.scatterplot(
    data=agrupado, 
    x='tiempo_respuesta', 
    y='nota_global_juez', 
    hue='modelo_usado', 
    s=300, # Puntos bien grandes
    palette='Set1',
    edgecolor='black',
    linewidth=1.5,
    ax=ax
)

# Añadir el nombre del modelo al lado de cada punto para que no haga falta mirar la leyenda
for i in range(agrupado.shape[0]):
    plt.text(
        agrupado['tiempo_respuesta'][i] + 0.1, # Desplazamiento X
        agrupado['nota_global_juez'][i],       # Altura Y
        agrupado['modelo_usado'][i], 
        horizontalalignment='left', 
        size='medium', color='black', weight='semibold'
    )

ax.set_xlabel("Tiempo Medio de Respuesta (segundos)")
ax.set_ylabel("Calidad Pedagógica (Nota Media del Juez)")

# Como hemos puesto los nombres en los puntos, ocultamos la leyenda para que quede más limpio
ax.get_legend().remove()

plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig(FIG_DIR / "scatter_calidad_vs_tiempo.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# TABLA DE CRIBA DE MODELOS
# ==========================================

# Métrica 1: Calidad Pedagógica Global
columnas_juez = score_cols
df['calidad_juez'] = df[columnas_juez].mean(axis=1)

# Métrica 3: Integración de los enunciados
df['integracion_media'] = df[['juez_medio_integracion', 'juez_bajo_integracion']].mean(axis=1)

# Métrica 4: Similitud Semántica
df['sim_pistas'] = df[['sim_pista_1', 'sim_pista_2']].mean(axis=1)
df['sim_enunciados'] = df[['sim_enunciado_medio', 'sim_enunciado_bajo']].mean(axis=1)

tabla_criba = df.groupby('modelo_usado').agg(
    Calidad_Pedagogica_Global=('calidad_juez', 'mean'),
    Fidelidad=('juez_fidelidad', 'mean'),
    Integracion_Enunciados=('integracion_media', 'mean'),
    Sim_Pistas=('sim_pistas', 'mean'),
    Sim_Enunciados=('sim_enunciados', 'mean'),
    Tiempo_Medio_s=('tiempo_respuesta', 'mean')
).reset_index()

def clasificar_velocidad(t):
    if pd.isna(t):
        return "Desconocido"
    elif t < 15:
        return "Rápido"
    elif t <= 60:
        return "Aceptable"
    else:
        return "Lento"

tabla_criba['Velocidad'] = tabla_criba['Tiempo_Medio_s'].apply(clasificar_velocidad)
tabla_criba = tabla_criba.sort_values(by='Calidad_Pedagogica_Global', ascending=False)

columnas_a_redondear = [
    'Calidad_Pedagogica_Global', 'Fidelidad', 'Integracion_Enunciados',
    'Sim_Pistas', 'Sim_Enunciados', 'Tiempo_Medio_s'
]
tabla_criba[columnas_a_redondear] = tabla_criba[columnas_a_redondear].round(2)

display(tabla_criba)


In [ ]:
# ==========================================
# BOXPLOT DE NOTAS GLOBALES POR MODELO
# ==========================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Columnas de evaluación del juez
score_cols = [
    'juez_p1_andamiaje', 'juez_p1_coherencia', 
    'juez_p2_andamiaje', 'juez_p2_coherencia',
    'juez_medio_integracion', 'juez_bajo_integracion', 
    'juez_fidelidad'
]

# Asegurar que las columnas son numéricas
for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Calcular la nota global si no existe ya
if 'nota_global_juez' not in df.columns:
    df['nota_global_juez'] = df[score_cols].mean(axis=1)

# Ordenar modelos por mediana de nota global
orden_modelos = (
    df.groupby('modelo_usado')['nota_global_juez']
    .median()
    .sort_values(ascending=False)
    .index
)

# Crear figura
fig, ax = plt.subplots(figsize=(10, 6))

# Boxplot principal
sns.boxplot(
    data=df,
    x='nota_global_juez',
    y='modelo_usado',
    order=orden_modelos,
    palette='viridis',
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 7
    },
    ax=ax
)

# Puntos individuales para ver la dispersión por tarea
sns.stripplot(
    data=df,
    x='nota_global_juez',
    y='modelo_usado',
    order=orden_modelos,
    color='black',
    alpha=0.35,
    size=4,
    jitter=0.18,
    ax=ax
)

# Etiquetas y formato
ax.set_xlabel("Nota Global del Juez (1-5)")
ax.set_ylabel("Modelo Evaluado")
ax.set_xlim(1, 5)

ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(FIG_DIR / "boxplot_notas_globales_por_modelo.png", dpi=300, bbox_inches="tight")
plt.show()